# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Sreenish Varakkoth · Week 5 Graded Mini Project**

Compares four prompting strategies — zero-shot, few-shot, structured/role-based
and chain-of-thought — on the same structured-extraction task: pull `company`,
`role` and `years_experience_required` out of a job-posting snippet.

Ten snippets × four strategies = 40 `gpt-4o-mini` calls, run concurrently, then
scored three ways against a golden set: exact-match accuracy, parse success, and
a `gpt-4o` LLM-as-judge rubric score. Cost and latency are captured per call.

Run order: top to bottom. Requires `OPENAI_API_KEY` in the environment.

---

## Setup

In [1]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
# Works whether the data sits beside the notebook (submission layout)
# or one level up (original course-package layout).
DATA_DIR = Path('data') if Path('data').is_dir() else Path('../data')


def load_jsonl(path: Path) -> list[dict]:
    """Read a JSON-Lines file into a list of dicts, skipping blank lines."""
    return [json.loads(line)
            for line in path.read_text(encoding='utf-8').splitlines()
            if line.strip()]


snippets = load_jsonl(DATA_DIR / 'job_snippets.jsonl')
golden = {row['id']: row for row in load_jsonl(DATA_DIR / 'golden_set.jsonl')}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries from {DATA_DIR.resolve()}')
print('Sample snippet:', snippets[0])


Loaded 10 snippets, 10 golden entries from C:\REPOS\IITM\Week5\Week 5_Graded Mini Project_Varakkoth\data
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [3]:
TASK_INSTRUCTION = (
    'Extract the company name, the role title, and the minimum years of '
    'experience required from the job posting below. '
    'Respond with only a JSON object with exactly these keys: '
    '"company" (string), "role" (string), '
    '"years_experience_required" (integer, or null if no years are stated).'
)


def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [
        {
            'role': 'user',
            'content': f'{TASK_INSTRUCTION}\n\nJob posting:\n{snippet_text}',
        }
    ]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    return [
        {
            'role': 'user',
            'content': f'{TASK_INSTRUCTION}\n\nJob posting:\nGlobex Inc. is looking for a Data Analyst to join our BI team. Candidates should have at least 3 years of experience with SQL and dashboarding tools.',
        },
        {
            'role': 'assistant',
            'content': '{"company": "Globex Inc.", "role": "Data Analyst", "years_experience_required": 3}',
        },
        {
            'role': 'user',
            'content': f'{TASK_INSTRUCTION}\n\nJob posting:\nInitech wants a passionate Product Designer to craft delightful user experiences alongside our founding team.',
        },
        {
            'role': 'assistant',
            'content': '{"company": "Initech", "role": "Product Designer", "years_experience_required": null}',
        },
        {
            'role': 'user',
            'content': f'{TASK_INSTRUCTION}\n\nJob posting:\n{snippet_text}',
        },
    ]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    return [
        {
            'role': 'system',
            'content': (
                'You are an expert technical recruiter who extracts structured data '
                'from job postings with perfect precision.\n\n'
                'For every job posting the user sends, respond with only a JSON object '
                'conforming exactly to this schema:\n'
                '{\n'
                '  "company": string,              // the hiring company\'s name, as written\n'
                '  "role": string,                 // the job title being hired for\n'
                '  "years_experience_required": integer | null   // minimum years required; null if not stated\n'
                '}\n\n'
                'Rules:\n'
                '- Output the JSON object only — no prose, no code fences.\n'
                '- Never invent values. If a field is not stated in the posting, use null.\n'
                '- For ranges like "3-5 years", use the minimum (3). For "5+ years", use 5.'
            ),
        },
        {
            'role': 'user',
            'content': f'Job posting:\n{snippet_text}',
        },
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [
        {
            'role': 'user',
            'content': (
                'Extract the company name, the role title, and the minimum years of '
                'experience required from the job posting below.\n\n'
                'Think step by step first:\n'
                '1. Identify the hiring company.\n'
                '2. Identify the role title being hired for.\n'
                '3. Look for an experience requirement and decide the minimum years, '
                'or null if no years are stated.\n\n'
                'Write out your reasoning, then on the final line give your answer as a '
                'JSON object with exactly these keys: "company" (string), "role" (string), '
                '"years_experience_required" (integer or null).\n\n'
                f'Job posting:\n{snippet_text}'
            ),
        }
    ]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [4]:
import re

from openai import APIConnectionError, APITimeoutError, RateLimitError

MAX_CONCURRENCY = 8   # cap simultaneous API calls to stay under rate limits
MAX_RETRIES = 4

_semaphore = asyncio.Semaphore(MAX_CONCURRENCY)


async def call_llm(model: str, messages: list[dict]) -> dict:
    """One chat completion with bounded concurrency and retry on transient errors.

    Returns {'text', 'prompt_tokens', 'completion_tokens', 'cost_usd', 'latency_s'}.
    """
    async with _semaphore:
        for attempt in range(MAX_RETRIES):
            try:
                start = time.perf_counter()
                response = await client.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=TEMPERATURE,
                )
                latency_s = time.perf_counter() - start
                usage = response.usage
                return {
                    'text': response.choices[0].message.content or '',
                    'prompt_tokens': usage.prompt_tokens,
                    'completion_tokens': usage.completion_tokens,
                    'cost_usd': (usage.prompt_tokens * RATES[model]['in']
                                 + usage.completion_tokens * RATES[model]['out']),
                    'latency_s': latency_s,
                }
            except (RateLimitError, APIConnectionError, APITimeoutError):
                if attempt == MAX_RETRIES - 1:
                    raise
                await asyncio.sleep(2 ** attempt)   # 1s, 2s, 4s


def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.

    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    if not text:
        return None

    text = text.strip()

    # Strip markdown fences if the whole response is wrapped in them
    if text.startswith('```'):
        text = text.removeprefix('```json').removeprefix('```')
        text = text.removesuffix('```').strip()

    # Try the whole text first (zero-shot / structured return bare JSON).
    # Fall back to the last {...} chunk (CoT puts its JSON after the reasoning).
    for candidate in (text, text[text.rfind('{') : text.rfind('}') + 1]):
        try:
            obj = json.loads(candidate)
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    return None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    messages = STRATEGIES[strategy_name](snippet['snippet'])
    call = await call_llm(MODEL, messages)
    return {
        'strategy': strategy_name,
        'snippet_id': snippet['id'],
        'raw_response': call['text'],
        'extracted': parse_response(call['text']),
        'prompt_tokens': call['prompt_tokens'],
        'completion_tokens': call['completion_tokens'],
        'cost_usd': call['cost_usd'],
        'latency_s': call['latency_s'],
    }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = [
        run_one(strategy_name, snippet)
        for strategy_name in STRATEGIES
        for snippet in snippets
    ]
    return await asyncio.gather(*tasks)

In [5]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}\n```',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'prompt_tokens': 108,
 'completion_tokens': 34,
 'cost_usd': 3.6599999999999995e-05,
 'latency_s': 1.8264565999852493}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [6]:
FIELDS = ['company', 'role', 'years_experience_required']

# Accept common field-name variants, in case a strategy renames a key (FAQ: normalise).
FIELD_ALIASES = {
    'company': ['company', 'company_name', 'employer'],
    'role': ['role', 'role_title', 'title', 'job_title'],
    'years_experience_required': [
        'years_experience_required', 'years_required',
        'years_of_experience', 'min_years_experience',
    ],
}


def get_field(extracted: dict, field: str):
    """Fetch a field, tolerating known field-name variants."""
    for alias in FIELD_ALIASES[field]:
        if alias in extracted:
            return extracted[alias]
    return None


def _norm(value):
    """Normalise a field for comparison.

    Strings: trim, then treat a leading number as an int so '5', '5+' and
    '5+ years' all normalise to 5 (FAQ: '5+' must match golden 5).
    Everything else lowercases. Ints/None pass through unchanged.
    """
    if isinstance(value, str):
        value = value.strip()
        head = value.split()[0].rstrip('+') if value.split() else ''
        if head.isdigit():
            return int(head)
        return value.lower()
    return value


def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    if extracted is None:
        return 0
    return sum(_norm(get_field(extracted, f)) == _norm(gold[f]) for f in FIELDS)


def score_fields(extracted: dict | None, gold: dict) -> dict:
    """Per-field correctness — shows *which* field a strategy gets wrong."""
    if extracted is None:
        return {f: False for f in FIELDS}
    return {f: _norm(get_field(extracted, f)) == _norm(gold[f]) for f in FIELDS}


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict):
    """Use gpt-4o as a judge. Returns (score 1-4, judge_cost_usd)."""
    if extracted is None:
        return 1, 0.0   # unparsable — rubric says 1, no judge call needed

    prompt = (
        'You are grading a data-extraction attempt against a gold answer.\n\n'
        f'Job posting:\n{snippet_text}\n\n'
        f'Gold answer:\n{json.dumps({f: gold[f] for f in FIELDS})}\n\n'
        f'Extracted answer:\n{json.dumps(extracted)}\n\n'
        'Rubric:\n'
        '4 - all three fields correct\n'
        '3 - two of three correct, no fabricated data\n'
        '2 - one of three correct, or fabricated a field\n'
        '1 - none correct\n\n'
        'Fabricated means the extracted value states something the posting does not '
        'support (e.g. a years number when the posting mentions none). Minor wording '
        "differences (case, punctuation, '5' vs '5+', 'Corp' vs 'Corporation') count "
        'as correct.\n\n'
        'Respond with only a single integer: 1, 2, 3, or 4.'
    )

    call = await call_llm(JUDGE_MODEL, [{'role': 'user', 'content': prompt}])

    match = re.search(r'[1-4]', call['text'])
    score = int(match.group()) if match else 1
    return score, call['cost_usd']

In [7]:
# Apply scoring to all 40 results
snippet_by_id = {s['id']: s['snippet'] for s in snippets}


async def score_row(row: dict) -> dict:
    """Attach accuracy, parse_success, per-field flags and the judge score to one result row."""
    gold = golden[row['snippet_id']]
    judge_score, judge_cost = await score_llm_judge(
        snippet_by_id[row['snippet_id']], row['extracted'], gold)
    fields_ok = score_fields(row['extracted'], gold)
    return {
        **row,
        'accuracy': score_accuracy(row['extracted'], gold),
        'parse_success': row['extracted'] is not None,
        'llm_judge_score': judge_score,
        'judge_cost_usd': judge_cost,
        **{f'ok_{f}': ok for f, ok in fields_ok.items()},
    }


scored = await asyncio.gather(*(score_row(row) for row in results))

worker_cost = sum(r['cost_usd'] for r in scored)
judge_cost = sum(r['judge_cost_usd'] for r in scored)
print(f'Scored {len(scored)} results.')
print(f'Worker cost (40 gpt-4o-mini calls): ${worker_cost:.4f}')
print(f'Judge cost  (40 gpt-4o calls):      ${judge_cost:.4f}')
print(f'Total spend this run:               ${worker_cost + judge_cost:.4f}')

Scored 40 results.
Worker cost (40 gpt-4o-mini calls): $0.0028
Judge cost  (40 gpt-4o calls):      $0.0236
Total spend this run:               $0.0264


In [8]:
# Persist the raw scored rows so the analysis (and any re-grading) can be redone
# without spending another 80 API calls.
Path('results.json').write_text(json.dumps(scored, indent=2), encoding='utf-8')
print(f'Wrote results.json ({len(scored)} rows)')


Wrote results.json (40 rows)


## Step 5 — Build the comparison table

In [9]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg(
    accuracy_mean=('accuracy', 'mean'),
    parse_rate=('parse_success', 'mean'),
    judge_score=('llm_judge_score', 'mean'),
    total_cost=('cost_usd', 'sum'),
    latency_p50=('latency_s', 'median'),
    out_tokens=('completion_tokens', 'mean'),
)

# Accuracy also as a 0-1 rate (out of 3 fields), and cost in cents for readability
summary.insert(1, 'accuracy_rate', (summary['accuracy_mean'] / 3))
summary['cost_cents'] = summary['total_cost'] * 100

summary = summary.round({
    'accuracy_mean': 2, 'accuracy_rate': 3, 'parse_rate': 2,
    'judge_score': 2, 'latency_p50': 2, 'out_tokens': 1, 'cost_cents': 4,
}).drop(columns='total_cost')

summary.columns = ['Accuracy (of 3)', 'Accuracy rate', 'Parse rate', 'Judge score',
                   'Latency p50 (s)', 'Mean out-tokens', 'Total cost (cents)']

# Order best-to-worst on accuracy so the winner reads off the top
summary = summary.sort_values('Accuracy rate', ascending=False)
summary

,Accuracy (of 3),Accuracy rate,Parse rate,Judge score,Latency p50 (s),Mean out-tokens,Total cost (cents)
strategy,,,,,,,
few_shot,2.8,0.933,1.0,4.0,0.65,25.6,0.0684
cot,2.7,0.900,1.0,3.9,1.92,170.4,0.1263
structured,2.7,0.900,1.0,3.9,0.90,30.5,0.0491
zero_shot,2.7,0.900,1.0,3.9,1.16,34.5,0.0374


### Per-field accuracy — which field actually breaks?

The aggregate 0-3 score hides *which* field fails. Splitting it out shows the
task is not uniformly hard: `company` and `role` are near-solved, and almost all
of the remaining error sits in `years_experience_required`.

In [10]:
field_cols = [f'ok_{f}' for f in FIELDS]
per_field = df.groupby('strategy')[field_cols].mean().round(2)
per_field.columns = [c.removeprefix('ok_') for c in per_field.columns]
per_field = per_field.loc[summary.index]   # same best-to-worst order as the summary
per_field


,company,role,years_experience_required
strategy,,,
few_shot,0.8,1.0,1.0
cot,0.8,1.0,0.9
structured,0.8,1.0,0.9
zero_shot,0.8,1.0,0.9


### Failure analysis — the j10 null trap

`j10` states no years requirement, so the correct answer is `null`. It is the
canonical hallucination probe: a weak prompt invents a plausible number rather
than reporting absence.

In [11]:
# j10 is the deliberate trap: the posting states no years requirement, so gold is null.
# A weak prompt fabricates a number here rather than admitting the data is absent.
print('j10 snippet:')
print(' ', snippet_by_id['j10'], '\n')
print('j10 gold:', {f: golden['j10'][f] for f in FIELDS}, '\n')

for _, r in df[df['snippet_id'] == 'j10'].sort_values('strategy').iterrows():
    got = (r['extracted'] or {}).get('years_experience_required', '<unparsed>')
    verdict = 'correct null' if got is None else f'FABRICATED {got!r}'
    print(f"  {r['strategy']:<11} {verdict:<22} "
          f"accuracy={r['accuracy']}/3  judge={r['llm_judge_score']}/4")

# Fabrication rate across every snippet whose gold years is null
null_ids = [i for i, g in golden.items() if g['years_experience_required'] is None]
null_rows = df[df['snippet_id'].isin(null_ids)]
fabrication_rate = (1 - null_rows.groupby('strategy')['ok_years_experience_required']
                    .mean()).round(2)

print(f'\nSnippets with gold years = null: {null_ids}')
print('Fabrication rate on those snippets (1.0 = always invented a number):')
print(fabrication_rate.to_string())


j10 snippet:
  Cyberdyne Systems is hiring. Role: AI/ML Research Scientist. PhD preferred but not required. Strong publication record in deep learning or reinforcement learning. We don't list a specific years requirement — we hire on demonstrated impact. 

j10 gold: {'company': 'Cyberdyne Systems', 'role': 'AI/ML Research Scientist', 'years_experience_required': None} 

  cot         correct null           accuracy=3/3  judge=4/4
  few_shot    correct null           accuracy=3/3  judge=4/4
  structured  correct null           accuracy=3/3  judge=4/4
  zero_shot   correct null           accuracy=3/3  judge=4/4

Snippets with gold years = null: ['j10']
Fabrication rate on those snippets (1.0 = always invented a number):
strategy
cot           0.0
few_shot      0.0
structured    0.0
zero_shot     0.0


### Where the judge disagrees with exact match

The judge sees meaning; exact match sees strings. Their disagreements are the
interesting rows — either the judge is being generous, or it caught something
strict matching missed.

In [12]:
# Where does the LLM judge disagree with strict exact-match scoring?
# accuracy 0-3 and judge 1-4 aren't the same scale, so compare them as rankings:
# a "generous" judge = full marks (4) on a row exact-match penalised (<3).
disagree = df[((df['llm_judge_score'] == 4) & (df['accuracy'] < 3))
              | ((df['llm_judge_score'] <= 2) & (df['accuracy'] == 3))].copy()

print(f'{len(disagree)} of {len(df)} rows where judge and exact match disagree materially.\n')

if len(disagree):
    for _, r in disagree.iterrows():
        gold = golden[r['snippet_id']]
        print(f"[{r['strategy']} / {r['snippet_id']}] accuracy={r['accuracy']}/3  judge={r['llm_judge_score']}/4")
        print(f"   gold: {({f: gold[f] for f in FIELDS})}")
        print(f"   got : {r['extracted']}\n")
else:
    print('The two measures agree on every row.')

8 of 40 rows where judge and exact match disagree materially.

[zero_shot / j02] accuracy=2/3  judge=4/4
   gold: {'company': 'Northwind Ltd', 'role': 'Data Analyst', 'years_experience_required': 2}
   got : {'company': 'Northwind Ltd.', 'role': 'Data Analyst', 'years_experience_required': 2}

[zero_shot / j08] accuracy=2/3  judge=4/4
   gold: {'company': 'Wonka Confectionery Ltd', 'role': 'Senior UX Researcher', 'years_experience_required': 5}
   got : {'company': 'Wonka Confectionery Ltd.', 'role': 'Senior UX Researcher', 'years_experience_required': 5}

[few_shot / j02] accuracy=2/3  judge=4/4
   gold: {'company': 'Northwind Ltd', 'role': 'Data Analyst', 'years_experience_required': 2}
   got : {'company': 'Northwind Ltd.', 'role': 'Data Analyst', 'years_experience_required': 2}

[few_shot / j08] accuracy=2/3  judge=4/4
   gold: {'company': 'Wonka Confectionery Ltd', 'role': 'Senior UX Researcher', 'years_experience_required': 5}
   got : {'company': 'Wonka Confectionery Ltd.', 'rol

### How much of the "error" is really a scoring artifact?

The disagreements above are all trailing-period cases. Re-scoring with a lenient
normaliser that ignores trailing punctuation separates *model* error from
*measurement* error — the gap between the two columns is error the model never
actually made.

In [13]:
def _norm_lenient(value):
    """Strict normalisation, plus: ignore trailing punctuation on strings."""
    value = _norm(value)
    return value.rstrip('.').strip() if isinstance(value, str) else value


def score_accuracy_lenient(extracted: dict | None, gold: dict) -> int:
    if extracted is None:
        return 0
    return sum(_norm_lenient(get_field(extracted, f)) == _norm_lenient(gold[f])
               for f in FIELDS)


df['accuracy_lenient'] = [
    score_accuracy_lenient(r['extracted'], golden[r['snippet_id']])
    for _, r in df.iterrows()
]

leniency = df.groupby('strategy').agg(
    strict=('accuracy', 'mean'),
    lenient=('accuracy_lenient', 'mean'),
).round(2)
leniency['artifact'] = (leniency['lenient'] - leniency['strict']).round(2)
leniency = leniency.loc[summary.index]
leniency.columns = ['Strict accuracy', 'Lenient accuracy', 'Punctuation artifact']
leniency


,Strict accuracy,Lenient accuracy,Punctuation artifact
strategy,,,
few_shot,2.8,3.0,0.2
cot,2.7,2.9,0.2
structured,2.7,2.9,0.2
zero_shot,2.7,2.9,0.2


### Every remaining miss, listed

With punctuation set aside, what is genuinely wrong?

In [14]:
residual = df[df['accuracy_lenient'] < 3]

if len(residual) == 0:
    print('No residual errors: every field matched once punctuation is normalised.')
else:
    print(f'{len(residual)} of {len(df)} rows still imperfect after lenient scoring:\n')
    for _, r in residual.sort_values(['snippet_id', 'strategy']).iterrows():
        g = golden[r['snippet_id']]
        for f in FIELDS:
            got, want = get_field(r['extracted'] or {}, f), g[f]
            if _norm_lenient(got) != _norm_lenient(want):
                print(f"  {r['snippet_id']} / {r['strategy']:<11} {f}: "
                      f"got {got!r}, gold {want!r}")


3 of 40 rows still imperfect after lenient scoring:

  j05 / cot         years_experience_required: got None, gold 0
  j05 / structured  years_experience_required: got None, gold 0
  j05 / zero_shot   years_experience_required: got None, gold 0


### Export the comparison table

In [15]:
# Export the comparison table as mp1_comparison.md (a required deliverable),
# so the numbers in the write-up can never drift from the numbers in the run.
from datetime import datetime

lines = [
    '# MP1 · Comparison Table',
    '',
    f'*Generated from a live run on {datetime.now():%Y-%m-%d %H:%M}. '
    f'Model: `{MODEL}` · Judge: `{JUDGE_MODEL}` · temperature={TEMPERATURE} · '
    f'{len(snippets)} snippets x {len(STRATEGIES)} strategies = {len(scored)} calls.*',
    '',
    '## Headline metrics by strategy',
    '',
    summary.to_markdown(),
    '',
    '## Per-field accuracy (share of 10 snippets correct)',
    '',
    per_field.to_markdown(),
    '',
    '## Strict vs lenient accuracy',
    '',
    leniency.to_markdown(),
    '',
    '*Lenient scoring ignores trailing punctuation. The gap is measurement error, not model error.*',
    '',
    '## Spend',
    '',
    f'- Worker calls (40 x `{MODEL}`): **${worker_cost:.4f}**',
    f'- Judge calls (`{JUDGE_MODEL}`): **${judge_cost:.4f}**',
    f'- Total: **${worker_cost + judge_cost:.4f}**',
    '',
    '## Column notes',
    '',
    '- **Accuracy (of 3)** — mean count of exactly-matching fields per snippet (0-3).',
    '- **Accuracy rate** — the same figure divided by 3, as a 0-1 rate.',
    '- **Parse rate** — share of responses from which a JSON object could be recovered.',
    '- **Judge score** — mean `gpt-4o` rubric score, 1-4.',
    '- **Latency p50** — median wall-clock seconds per call, measured inside the '
    'concurrency semaphore (queue wait excluded).',
    '- **Total cost** — worker-call spend only; judge spend is listed separately above.',
    '',
]

Path('mp1_comparison.md').write_text('\n'.join(lines), encoding='utf-8')
print('Wrote mp1_comparison.md')

Wrote mp1_comparison.md

## Step 6 — Reflection

The one-page reflection lives in `mp1_writeup.md`, and the generated metrics
table in `mp1_comparison.md` (written by the export cell above).

See `README.md` for how to run this notebook and how to read the numbers.